In [1]:
import joblib
import pandas as pd
import numpy as np


 Load processed_data & trained model

In [2]:
df=pd.read_csv('../data/processed_telco_churn.csv')
lr= joblib.load("../models/customer_churn_model.pkl")
X=df.drop('Churn',axis=1)

Predict churn

In [3]:
df["predicted_churn"] = lr.predict(X)


 Predict churn probability

In [4]:
df["Churn_Probability"] = lr.predict_proba(X)[:, 1]

Calculate CLV, Priority Score

In [ ]:
#Calculate CLV
df["CLV"]=df["MonthlyCharges"]*df["tenure"]
# Priority Score (0–100)
df["Priority_Score"] = (
    df["Churn_Probability"] * 0.7 +
    (df["CLV"] / df["CLV"].max()) * 0.3
) * 100

df["Priority_Score"] = df["Priority_Score"].round(2)

8550.0


Assign Priority Level to each customer

In [6]:
def priority_level(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

df["Priority_Level"] = df["Priority_Score"].apply(priority_level)


Create Smart Retention Recommendation

In [7]:
def smart_retention(row):

    priority = row["Priority_Level"]
    clv = row["CLV"]
    tenure = row["tenure"]
    monthly = row["MonthlyCharges"]

    # Critical Priority
    if priority == "Critical":

        if clv >= 5000:
            return "Immediate Call + 30% Discount + Dedicated Relationship Manager"

        elif tenure < 12:
            return "Free 2-Month Subscription + Onboarding Support"

        elif monthly >= 80:
            return "25% Discount + Premium Support"

        else:
            return "20% Discount + Loyalty Rewards"

    # High Priority
    elif priority == "High":

        if clv >= 5000:
            return "Premium Support + Service Upgrade"

        elif monthly >= 80:
            return "20% Discount"

        else:
            return "15% Discount + Loyalty Points"

    # Medium Priority
    elif priority == "Medium":

        if tenure < 12:
            return "Welcome Offer + Personalized Email"

        else:
            return "10% Discount + Plan Upgrade Recommendation"

    # Low Priority
    else:

        if clv >= 5000:
            return "Loyalty Rewards + Thank You Coupon"

        else:
            return "Regular Promotional Email"

df["Retention_Action"] = df.apply(smart_retention, axis=1)     

In [8]:

retention_report = df[[
    "Churn_Probability",
    "CLV",
    "Priority_Score",
    "Priority_Level",
    "Retention_Action"
]].sort_values(
    by="Priority_Score",
    ascending=False
)

retention_report.head(20)

,Churn_Probability,CLV,Priority_Score,Priority_Level,Retention_Action
6380,0.709636,6782.75,73.47,High,Premium Support + Service Upgrade
2879,0.688112,7206.50,73.45,High,Premium Support + Service Upgrade
4181,0.731362,6283.70,73.24,High,Premium Support + Service Upgrade
2075,0.762093,5658.80,73.20,High,Premium Support + Service Upgrade
4074,0.753724,5724.60,72.85,High,Premium Support + Service Upgrade
3330,0.790399,4987.80,72.83,High,20% Discount
3200,0.756834,5632.20,72.74,High,Premium Support + Service Upgrade
7023,0.709686,6520.50,72.56,High,Premium Support + Service Upgrade
6032,0.677243,7160.40,72.53,High,Premium Support + Service Upgrade
2609,0.712638,6438.55,72.48,High,Premium Support + Service Upgrade


In [9]:
df.to_csv("customer_retention_result.csv", index=False)

print("Retention results saved successfully!")

Retention results saved successfully!
